# Escalation Patterns - Knowing When to Involve Humans

AI agents are fast and consistent within their operational range, but every agent has limits - queries whose correct answer it cannot determine reliably, decisions that require human authority, or situations where the cost of a wrong response is too high to risk. The challenge is not just having those limits but recognising them at runtime: escalating to a human with full context rather than producing a confidently wrong answer.

This notebook demonstrates how to build intelligent escalation into a LangGraph agent workflow. We build a customer-service agent that analyses each incoming query, self-assesses whether it can handle the query reliably, and either responds autonomously or prepares a structured handoff to the appropriate human team. The escalation decision combines a confidence score with an explicit flag the LLM sets when it identifies a category-level reason for human handling - policy exceptions, safety concerns or explicit customer requests.

In [1]:
import os
import uuid
from typing import TypedDict, Literal, Optional, List, Dict, Any
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END

### Initialize the language model

In [2]:
# Lower temperature produces more consistent confidence scores and escalation flags
llm = ChatOpenAI(model='gpt-4o-mini', api_key=os.getenv("OPENAI_API_KEY", "").strip(), temperature=0.3)

## Understanding escalation patterns
Escalation is the mechanism by which an agent transfers control to a human when the situation exceeds its reliable operating range. The key insight is that this transfer should be deliberate and informed rather than a failure state. An agent that escalates with complete context, a clear reason, and a priority assessment makes the human's job straightforward. An agent that escalates only after producing a confused answer - or that never escalates and guesses through difficult queries - undermines both efficiency and trust.

There are four distinct reasons an agent should escalate:
- Low confidence occurs when the agent genuinely cannot determine a reliable answer, which it detects through self-assessment during generation.
- Policy exceptions arise when a query requires authority the agent does not have - waiving a fee, granting a refund outside policy, or making a commitment on behalf of the organisation.
- Safety concerns emerge when the query involves potential harm, harassment, or legally sensitive topics where a wrong response could have serious consequences.
- Customer requests are the simplest trigger: the customer has explicitly asked to speak to a human.

The lifecycle has three stages. The analysis stage combines response generation with a self-assessment that produces a confidence score and an escalation flag. The decision stage applies deterministic rules: if the flag is set or confidence falls below a threshold, escalate. The handoff or delivery stage either routes to the appropriate human team with a structured context package, or delivers the agent's response directly to the customer.

### Define the escalation state
The state structure reflects this three-stage lifecycle directly. Input fields are set once at invocation. Analysis fields are written by the generation node. Routing fields are written by the decision node. Handoff fields are written only when an escalation actually occurs.

In [3]:
class EscalationState(TypedDict):
    '''State for the escalation workflow.

    Progresses through three stages: input fields are set at invocation, analysis fields are written by analyze_query, routing fields by decide_escalation, and handoff fields only when escalating.
    '''
    # Input -- set once at invocation, never modified by any node
    customer_query: str
    customer_context: Optional[Dict[str, Any]]        # Account info, tier, history
    conversation_history: Optional[List[Dict[str, str]]]  # Previous turns, if any

    # Analysis -- written by analyze_query
    agent_response: Optional[str]          # Agent's draft answer to the query
    confidence_score: Optional[float]      # 0.0--1.0; below 0.5 triggers escalation
    escalation_flag: Optional[Literal[     # Explicit reason from the LLM; None = no issue
        'low_confidence',    # Agent is uncertain about the answer
        'policy_exception',  # Requires human authority to decide
        'safety_concern',    # Sensitive or potentially harmful topic
        'customer_request'   # Customer explicitly asked to speak to a human
    ]]

    # Routing -- written by decide_escalation
    should_escalate: bool
    priority: Optional[Literal['low', 'medium', 'high', 'urgent']]
    assigned_team: Optional[str]           # Which human team receives the escalation

    # Handoff -- written by prepare_handoff only when escalating
    escalation_id: Optional[str]           # Unique ID for tracking through to resolution
    escalation_summary: Optional[str]      # Formatted context package for the human agent

The `EscalationState` separates concerns cleanly across the three workflow stages.
- The input fields (`customer_query`, `customer_context`, `conversation_history`) are set at invocation and are never modified. All three are available to every node that needs them.
- `escalation_flag` is the most important design decision in the state. By having the LLM set it explicitly during generation, the flag reflects the model's actual understanding of why the query warrants human handling - rather than a post-hoc keyword scan that could miss paraphrases or generate false positives on unrelated text.
- The routing fields (`should_escalate`, `priority`, `assigned_team`) are written by a separate deterministic function after the LLM returns. Keeping routing logic deterministic makes it auditable and tunable without touching the prompt.
- `escalation_id` and `escalation_summary` are only populated for escalated interactions. Leaving them as `None` on the autonomous path keeps the state clean and signals clearly that no handoff occurred.

### Analyze the query
The analysis node is the only LLM-heavy step in the workflow. Its job is to generate a response and simultaneously produce a structured self-assessment. Asking the LLM to output an explicit escalation flag - rather than just a confidence score - is more reliable because the model can recognise category-level reasons for escalation (safety, policy authority) that a low number alone would not convey.

In [4]:
def analyze_query(state: EscalationState) -> dict:
    '''Generate a response and self-assess confidence and escalation need.

    Asks the LLM to set an explicit escalation flag rather than relying on post-hoc keyword matching. This grounds the escalation signal in the model's own assessment of whether it can handle the query reliably.
    '''
    query = state['customer_query']
    context = state.get('customer_context') or {}
    history = state.get('conversation_history') or []

    # Build a compact history block to include in the prompt
    history_text = ''
    if history:
        history_text = '\n\nConversation so far:\n'
        for turn in history[-4:]:   # cap at 4 turns to avoid bloating the prompt
            history_text += turn.get('role', '?') + ': ' + turn.get('content', '') + '\n'

    # Build the prompt as a joined list of lines to keep the structure readable
    context_str = context if context else 'none'
    prompt_lines = [
        'You are a customer service agent. Analyse this query and respond.',
        '',
        f'Customer query: {query}',
        f'Context: {context_str}{history_text}',
        '',
        'Reply in this exact format:',
        '',
        'RESPONSE:',
        '[Your best answer to the customer]',
        '',
        'CONFIDENCE:',
        '[A decimal 0.0-1.0. Score below 0.5 if genuinely uncertain.]',
        '',
        'ESCALATE:',
        '[Exactly one of: none / low_confidence / policy_exception / safety_concern / customer_request]',
        '- none             : confident, no special handling needed',
        '- low_confidence   : uncertain; a human would give a better answer',
        '- policy_exception : requires human authority (fee waiver, policy override, exception)',
        '- safety_concern   : involves harm, harassment, or legal risk',
        '- customer_request : customer explicitly asked to speak to a human',
        '',
        'REASONING:',
        '[One sentence explaining your confidence and escalation choice]',
    ]
    prompt = '\n'.join(prompt_lines)

    result = llm.invoke(prompt)
    content = result.content

    # Parse the four labelled sections sequentially
    try:
        after_response = content.split('CONFIDENCE:')
        response = after_response[0].replace('RESPONSE:', '').strip()

        after_confidence = after_response[1].split('ESCALATE:')
        confidence = float(after_confidence[0].strip())

        after_escalate = after_confidence[1].split('REASONING:')
        flag_raw = after_escalate[0].strip().lower()
        reasoning = after_escalate[1].strip() if len(after_escalate) > 1 else ''

        # Accept only the defined flag values; treat 'none' and anything else as None
        valid_flags = {'low_confidence', 'policy_exception', 'safety_concern', 'customer_request'}
        escalation_flag = flag_raw if flag_raw in valid_flags else None

    except (IndexError, ValueError):
        # Parse failure: default to below-threshold confidence so the decision
        # node routes toward human review rather than autonomous handling
        response = content.strip()
        confidence = 0.4
        escalation_flag = 'low_confidence'
        reasoning = 'Parse failure -- defaulting to escalation'

    flag_label = escalation_flag or 'none'
    print(f"\n\U0001f50d Analysis  |  Confidence: {confidence:.2f}  |  Flag: {flag_label}")
    print(f"   {reasoning}")

    return {
        'agent_response': response,
        'confidence_score': confidence,
        'escalation_flag': escalation_flag,
    }

`analyze_query` implements the analysis stage in a single LLM call with four labelled output sections.
- The prompt is built as a list of strings joined with `\n` rather than a triple-quoted block. This keeps the indentation clean in the notebook and avoids the need for continuation backslashes.
- The four sections - `RESPONSE:`, `CONFIDENCE:`, `ESCALATE:`, `REASONING:` - are parsed by sequential string splits. Each split is independent, so a missing section only affects the fields parsed after it, rather than crashing the entire parse chain.
- The `ESCALATE:` value is validated against a fixed `valid_flags` set. Anything outside that set, including the literal string `'none'`, maps to `None`. This means the flag is only set when the LLM explicitly identifies a category-level reason; ambiguous or non-matching responses do not falsely trigger escalation.
- The parse fallback sets `confidence = 0.4` rather than `1.0`. A failed parse signals unexpected model behaviour - the correct default is to route toward human review, not away from it.

### Make the escalation decision
The decision function is deterministic: it receives the analysis outputs from state and applies fixed mapping tables to produce the routing fields. Separating this logic from the LLM call means the routing rules can be unit-tested with fixed inputs, their thresholds can be adjusted without touching the prompt, and their reasoning is fully transparent in the code.

In [5]:
def decide_escalation(state: EscalationState) -> dict:
    '''Apply deterministic rules to the analysis outputs and set routing fields.

    Escalation is triggered by two independent conditions:
      1. The LLM explicitly set an escalation flag during analysis.
      2. Confidence falls below 0.5, even if the LLM left the flag empty.

    Condition 2 is a safety net for cases where the model under-flagged its uncertainty. Both conditions are evaluated independently and combined.
    '''
    confidence = state.get('confidence_score', 0.5)
    flag = state.get('escalation_flag')

    # Condition 1: LLM set an explicit flag
    # Condition 2: confidence is below the threshold even without an explicit flag
    if flag or confidence < 0.5:
        should_escalate = True
        reason = flag or 'low_confidence'  # assign a reason if triggered by confidence alone
    else:
        should_escalate = False
        reason = None

    # Each escalation reason maps to a priority level and a responsible human team
    priority_map = {
        'safety_concern':   'urgent',
        'policy_exception': 'high',
        'customer_request': 'high',
        'low_confidence':   'medium',
    }
    team_map = {
        'safety_concern':   'safety_team',
        'policy_exception': 'management',
        'customer_request': 'senior_support',
        'low_confidence':   'general_support',
    }

    priority = priority_map.get(reason) if should_escalate else None
    assigned_team = team_map.get(reason)    if should_escalate else None

    icon = '\U0001f6a8' if should_escalate else '\u2705'
    label = 'ESCALATE' if should_escalate else 'HANDLE AUTONOMOUSLY'
    print(f"\n{icon} Decision: {label}")
    if should_escalate:
        print(f"   Reason   : {reason}")
        print(f"   Priority : {priority.upper()}")
        print(f"   Team     : {assigned_team}")

    return {
        'should_escalate': should_escalate,
        'escalation_flag': reason,    # may set 'low_confidence' if triggered by confidence alone
        'priority': priority,
        'assigned_team': assigned_team,
    }

`decide_escalation` converts the two analysis signals into routing fields using lookup tables.
- The two escalation conditions are independent. The LLM flag takes priority, but the confidence threshold catches cases where the model produced a low score without explicitly flagging the reason.
- `escalation_flag` may be overwritten in the return dict. If the LLM left it as `None` but confidence fell below 0.5, the function sets it to `'low_confidence'` so the handoff node always has a human-readable reason to include in the summary.
- The `priority_map` and `team_map` dictionaries are the entire routing specification. Adding a new escalation category or reassigning a team means editing one entry in each dict, with no control-flow changes required elsewhere.
- Returning `None` for `priority` and `assigned_team` on the non-escalation path keeps those state fields clean and signals clearly to downstream logic that no handoff is occurring.

### Prepare the handoff package
When escalation is confirmed, the handoff node's sole responsibility is to build a context package that allows a human agent to understand the situation immediately without asking the customer to repeat themselves. A good handoff surfaces everything the agent observed - the query, confidence level, agent draft, and any conversation context - compactly enough that the human can act within seconds of opening the ticket.

In [6]:
def prepare_handoff(state: EscalationState) -> dict:
    '''Build a structured context package for the receiving human agent.

    Assembles everything the human needs to act immediately without additional LLM inference -- all required information is already in state.
    '''
    # Unique ID to track this escalation through ticketing systems and resolution logs
    escalation_id = f'ESC-{uuid.uuid4().hex[:8].upper()}'

    # Extract routing fields into local variables for clean f-string formatting
    reason   = state.get('escalation_flag') or 'unspecified'
    priority = (state.get('priority') or 'medium').upper()
    team     = state.get('assigned_team') or 'general_support'
    query    = state['customer_query']

    lines = [
        f'ESCALATION ID   : {escalation_id}',
        f'REASON          : {reason}',
        f'PRIORITY        : {priority}',
        f'TEAM            : {team}',
        '',
        f'QUERY           : {query}',
    ]

    if state.get('customer_context'):
        lines.append(f'CONTEXT         : {state["customer_context"]}')

    # Include conversation history if it was provided at invocation
    if state.get('conversation_history'):
        lines.append('')
        lines.append('CONVERSATION HISTORY:')
        for turn in (state['conversation_history'] or [])[-4:]:
            role    = turn.get('role', '?')
            content = turn.get('content', '')
            lines.append(f'  {role}: {content}')

    lines += [
        '',
        f'AGENT CONFIDENCE: {state.get("confidence_score", 0):.2f}',
        'AGENT DRAFT     :',
        f'  {state.get("agent_response", "none")}',
    ]

    summary = '\n'.join(lines)

    # Print the handoff package (in production: POST to ticket system or alert queue)
    print('\n' + '=' * 65)
    print('  ESCALATION HANDOFF')
    print('=' * 65)
    print(f'\n{summary}')
    print('\n' + '=' * 65)

    return {
        'escalation_id': escalation_id,
        'escalation_summary': summary,
    }

`prepare_handoff` assembles the context package from state without any additional LLM inference.
- The `uuid` module (imported at the notebook level) generates a collision-resistant ID. Generating it here rather than in `decide_escalation` keeps the ID creation co-located with the handoff output it labels.
- Routing fields are extracted into local variables before the `lines` list is built. This avoids embedding `state.get(...)` calls inside f-string expressions, which would require mixed quote styles and makes the format strings harder to read.
- Fixed-width labels in the summary string make it scannable. A human reviewing a queue of escalations needs to identify the reason and priority within seconds; consistent column alignment supports fast scanning.
- The agent's draft response is included under `AGENT DRAFT`. Surfacing it allows the human to accept it as a starting point, correct it, or set it aside - in all cases the draft reduces the time needed to write a response from scratch.
- Conversation history is capped at four turns, the same limit used in the analysis prompt, so the handoff remains compact even for long sessions.

### Handle the query autonomously
The autonomous path is the success case: confidence is high, no escalation flag was set, and the response can be delivered to the customer directly. This node is intentionally minimal - the substantive work happened in `analyze_query`. Its only responsibility is to deliver the response and signal that the interaction is complete.

In [7]:
def handle_autonomously(state: EscalationState) -> dict:
    '''Deliver the agent response directly; only reachable when no escalation is needed.

    In production this node would dispatch the response to the customer and log the interaction for quality monitoring.
    '''
    confidence = state.get('confidence_score') or 0.0
    response   = state.get('agent_response') or ''

    print('\n' + '=' * 65)
    print('  AUTONOMOUS RESPONSE')
    print('=' * 65)
    print(f'  Confidence : {confidence:.2f}')
    print(f'\n  Response   :\n  {response}')
    print('=' * 65)

    # Return an empty dict -- this terminal node writes no new state fields
    return {}

`handle_autonomously` is intentionally short. Its design is symmetric with `prepare_handoff`: each terminal node is a thin wrapper around its respective side effect (delivery vs. escalation), with no processing logic duplicated between them.
- Local variables `confidence` and `response` are extracted before the print statements so the display logic reads cleanly without inline `state.get(...)` calls.
- Returning `{}` is valid in LangGraph: merging an empty dict into state produces no changes. This node is a terminal node - it has nothing new to write to state.

### Build the escalation workflow graph
With all four node functions defined, the graph connects them through a single conditional edge. Every query passes through `analyze_query` and `decide_escalation` unconditionally before the routing function determines which terminal branch to take. This means the analysis cost is paid for every interaction, but the routing decision is always informed by a complete assessment rather than a shortcut.

In [8]:
def route_after_decision(state: EscalationState) -> str:
    '''Branch to escalation or autonomous handling based on the decision output.'''
    if state.get('should_escalate', False):
        return 'prepare_handoff'
    return 'handle_autonomously'


# Build the graph
workflow = StateGraph(EscalationState)

workflow.add_node('analyze_query',       analyze_query)
workflow.add_node('decide_escalation',   decide_escalation)
workflow.add_node('prepare_handoff',     prepare_handoff)
workflow.add_node('handle_autonomously', handle_autonomously)

# Every query flows through analysis and decision unconditionally
workflow.set_entry_point('analyze_query')
workflow.add_edge('analyze_query', 'decide_escalation')

# The conditional edge implements the routing fork after the decision is made
workflow.add_conditional_edges(
    'decide_escalation',
    route_after_decision,
    {
        'prepare_handoff':    'prepare_handoff',
        'handle_autonomously': 'handle_autonomously',
    }
)

# Both terminal nodes connect to END
workflow.add_edge('prepare_handoff',    END)
workflow.add_edge('handle_autonomously', END)

escalation_app = workflow.compile()

The graph structure maps directly to the three-stage lifecycle described at the start of the notebook.
- The unconditional edge from `analyze_query` to `decide_escalation` ensures every query is fully analysed before any routing decision is made. There is no early-exit path that skips the decision node.
- `route_after_decision` reads a single boolean field from state, making it the simplest possible routing function. All complexity lives in `decide_escalation`; the router just reads the result.
- Both `prepare_handoff` and `handle_autonomously` connect to `END`, producing a symmetric graph structure with two well-defined terminal paths that `workflow.compile()` validates as complete.
- `route_after_decision` is defined as a named function rather than a lambda so it is readable in isolation and straightforward to unit-test without running the full graph.

### Run an example
The initial state supplies the customer query and optional context. All fields written by workflow nodes start as `None` or `False`. Initialising every TypedDict field explicitly - including those the workflow will overwrite — makes the full state contract visible and prevents `KeyError` exceptions if a node reads a field before the node that writes it has run.

In [9]:
simple_state: EscalationState = {
    'customer_query': 'What are your business hours?',
    'customer_context': {'account_type': 'standard'},
    'conversation_history': None,
    # Analysis fields -- populated by analyze_query
    'agent_response': None,
    'confidence_score': None,
    'escalation_flag': None,
    # Routing fields -- populated by decide_escalation
    'should_escalate': False,
    'priority': None,
    'assigned_team': None,
    # Handoff fields -- populated by prepare_handoff when escalating
    'escalation_id': None,
    'escalation_summary': None,
}

print('\U0001f680 Test 1 -- simple factual query\n')
result1 = escalation_app.invoke(simple_state)

print('\n' + '=' * 65)
print('  TEST 1 SUMMARY')
print('=' * 65)
outcome = 'ESCALATED' if result1['should_escalate'] else 'HANDLED AUTONOMOUSLY'
print(f'  Outcome    : {outcome}')
print(f'  Confidence : {result1["confidence_score"]:.2f}')
print(f'  Flag       : {result1["escalation_flag"] or "none"}')

🚀 Test 1 -- simple factual query


🔍 Analysis  |  Confidence: 1.00  |  Flag: none
   I am confident in providing the standard business hours as they are typically consistent and widely known.

✅ Decision: HANDLE AUTONOMOUSLY

  AUTONOMOUS RESPONSE
  Confidence : 1.00

  Response   :
  Our business hours are Monday to Friday, from 9 AM to 5 PM.

  TEST 1 SUMMARY
  Outcome    : HANDLED AUTONOMOUSLY
  Confidence : 1.00
  Flag       : none


## Testing escalation triggers
A simple factual query is the baseline success case: the agent should answer with high confidence, no escalation flag should be set, and the workflow should deliver the response autonomously. The more revealing tests probe the two most consequential escalation categories.

The next test presents a query that spans multiple issues - a billing dispute and an explicit request to waive a cancellation fee. A waiver is a policy exception: the agent cannot grant it unilaterally, because doing so would involve making a financial commitment on behalf of the organisation without the authority to do so. The expected outcome is a high-priority handoff to the management team.

The final test presents a safety-concern scenario: a customer reporting threatening messages from another user. This is the most time-sensitive escalation category - a wrong or delayed response could cause real harm. The expected outcome is an urgent handoff to the safety team, with the agent's draft included in the context package as a starting point for the human's response.

In [10]:
# A query combining a billing dispute with an explicit fee-waiver request --
# granting a waiver requires management authority the agent does not have
policy_state: EscalationState = {
    'customer_query': (
        'I need to dispute a charge from 6 months ago and also close my account. '
        'Can you waive the cancellation fee? I have been a customer for two years.'
    ),
    'customer_context': {'account_type': 'premium', 'tenure_months': 24},
    'conversation_history': [
        {'role': 'customer', 'content': 'I have been having issues with billing'},
        {'role': 'agent',    'content': 'I am sorry to hear that. How can I help?'},
    ],
    'agent_response': None,
    'confidence_score': None,
    'escalation_flag': None,
    'should_escalate': False,
    'priority': None,
    'assigned_team': None,
    'escalation_id': None,
    'escalation_summary': None,
}

print('\U0001f680 Test 2 -- policy exception query\n')
result2 = escalation_app.invoke(policy_state)

print('\n' + '=' * 65)
print('  TEST 2 SUMMARY')
print('=' * 65)
outcome2 = 'ESCALATED' if result2['should_escalate'] else 'HANDLED AUTONOMOUSLY'
print(f'  Outcome    : {outcome2}')
print(f'  Flag       : {result2.get("escalation_flag") or "none"}')
print(f'  Priority   : {result2.get("priority") or "N/A"}')
print(f'  Team       : {result2.get("assigned_team") or "N/A"}')
print(f'  Escal. ID  : {result2.get("escalation_id") or "N/A"}')

🚀 Test 2 -- policy exception query


🔍 Analysis  |  Confidence: 0.80  |  Flag: policy_exception
   I am confident in addressing the dispute but need to escalate the cancellation fee waiver request as it requires human authority to approve exceptions to policy.

🚨 Decision: ESCALATE
   Reason   : policy_exception
   Priority : HIGH
   Team     : management

  ESCALATION HANDOFF

ESCALATION ID   : ESC-B5B382A8
REASON          : policy_exception
PRIORITY        : HIGH
TEAM            : management

QUERY           : I need to dispute a charge from 6 months ago and also close my account. Can you waive the cancellation fee? I have been a customer for two years.
CONTEXT         : {'account_type': 'premium', 'tenure_months': 24}

CONVERSATION HISTORY:
  customer: I have been having issues with billing
  agent: I am sorry to hear that. How can I help?

AGENT CONFIDENCE: 0.80
AGENT DRAFT     :
  I understand your concerns regarding the disputed charge and your request to close your account. Sinc

In [11]:
# A safety-concern query: harassment and threatening messages require immediate
# human handling -- any delay or wrong response could cause real harm
safety_state: EscalationState = {
    'customer_query': (
        'Another user has been sending me threatening messages on your platform '
        'and I feel unsafe. I need this handled immediately.'
    ),
    'customer_context': {'account_type': 'standard'},
    'conversation_history': None,
    'agent_response': None,
    'confidence_score': None,
    'escalation_flag': None,
    'should_escalate': False,
    'priority': None,
    'assigned_team': None,
    'escalation_id': None,
    'escalation_summary': None,
}

print('\U0001f680 Test 3 -- safety concern\n')
result3 = escalation_app.invoke(safety_state)

print('\n' + '=' * 65)
print('  TEST 3 SUMMARY')
print('=' * 65)
outcome3 = 'ESCALATED' if result3['should_escalate'] else 'HANDLED AUTONOMOUSLY'
print(f'  Outcome    : {outcome3}')
print(f'  Flag       : {result3.get("escalation_flag") or "none"}')
print(f'  Priority   : {result3.get("priority") or "N/A"}')
print(f'  Team       : {result3.get("assigned_team") or "N/A"}')
print(f'  Escal. ID  : {result3.get("escalation_id") or "N/A"}')

🚀 Test 3 -- safety concern


🔍 Analysis  |  Confidence: 1.00  |  Flag: safety_concern
   The nature of the customer's query involves threats and safety concerns, which necessitates immediate attention and escalation to ensure their safety.

🚨 Decision: ESCALATE
   Reason   : safety_concern
   Priority : URGENT
   Team     : safety_team

  ESCALATION HANDOFF

ESCALATION ID   : ESC-FE9427CB
REASON          : safety_concern
PRIORITY        : URGENT
TEAM            : safety_team

QUERY           : Another user has been sending me threatening messages on your platform and I feel unsafe. I need this handled immediately.
CONTEXT         : {'account_type': 'standard'}

AGENT CONFIDENCE: 1.00
AGENT DRAFT     :
  I'm very sorry to hear that you're experiencing this situation. Your safety is our top priority, and we take threats very seriously. Please provide us with the details of the messages and the user involved, and we will take immediate action to investigate and address this issue.


  TES

The three tests show the escalation system responding correctly across the full range of scenarios: confident autonomous handling for a simple query, a high-priority management handoff for a fee-waiver request, and an urgent safety-team handoff for a harassment report. In each case the agent's self-assessment drove the routing decision without any hard-coded content matching on the query text.

When to use escalation patterns:
- Customer service agents that handle queries of varying risk and complexity, where a single confidence threshold would either over-escalate routine queries or under-escalate edge cases.
- Decision-making agents where incorrect outputs carry real consequences - financial commitments, regulated communications, or safety-sensitive responses.
- Content moderation systems where edge cases require human judgment on community standards.
- Technical support pipelines with tiered difficulty levels, where senior agents should only see queries that genuinely require their expertise.

Design principles that matter in production:
- Ask the LLM to set an explicit escalation flag rather than inferring escalation from a confidence score alone - the flag captures category-level reasons (policy authority, safety) that a number cannot convey.
- Keep routing logic deterministic and separate from the LLM call so it can be tested, audited, and tuned without touching the prompt.
- Build the handoff package from state without additional LLM calls - all necessary information was captured during analysis.
- Default to below-threshold confidence on parse failures so unexpected model behaviour routes toward human review rather than away from it.
- Initialise all state fields explicitly at invocation to make the full state contract visible and prevent `KeyError` exceptions.

In production, escalation IDs should be written to an external tracking system - a ticketing platform, a database or an alert queue - rather than existing only in workflow state. Tracking resolution outcomes against escalation categories creates a feedback loop: if a category generates many escalations that humans resolve in a standard repeatable way, the agent can be improved to handle that category autonomously, progressively narrowing the escalation surface over time.